<a href="https://colab.research.google.com/github/fnjimenez/curso-analitica-datos/blob/main/VDRER_dataset_super_store_sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Configurar Google Colab

Dejar las configuraciones por default, colocar un nombre al Block de notas, Notebok de Colab Python

# 2. Cargar Dataset y Librerías

In [1]:
# ===================================================================
# BLOQUE 1: CONFIGURACIÓN INICIAL Y LIBRERÍAS
# ===================================================================
# Importar librerías esenciales para limpieza
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

# Configuración para mejor visualización
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configurar pandas para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Verificar instalación exitosa
print("✅ Librerías importadas exitosamente")
print(f"📊 Pandas versión: {pd.__version__}")
print(f"🔢 NumPy versión: {np.__version__}")
print(f"📈 Matplotlib versión: {plt.matplotlib.__version__}")

✅ Librerías importadas exitosamente
📊 Pandas versión: 2.2.2
🔢 NumPy versión: 2.0.2
📈 Matplotlib versión: 3.10.0


In [1]:
# ===================================================================
# BLOQUE 2: CARGA DE DATOS
# ===================================================================
# Opción 1: Subir archivo desde tu computadora
from google.colab import files

print("📂 Selecciona tu archivo CSV desde tu computadora:")
uploaded = files.upload()

# Obtener el nombre del archivo subido
filename = list(uploaded.keys())[0]
print(f"📄 Archivo detectado: {filename}")

# Cargar el dataset con manejo de errores
try:
    df = pd.read_csv(filename, encoding='utf-8')
except UnicodeDecodeError:
    print("⚠️ Problema de encoding, intentando con latin-1...")
    df = pd.read_csv(filename, encoding='latin-1')

# Información básica de carga
print(f"✅ Dataset cargado exitosamente")
print(f"📏 Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"💾 Memoria usada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Opción 2: Cargar desde URL (comentado)
# df = pd.read_csv('https://tu-url-del-dataset.csv')

📂 Selecciona tu archivo CSV desde tu computadora:


Saving super_store_sales.csv to super_store_sales.csv
📄 Archivo detectado: super_store_sales.csv
✅ Dataset cargado exitosamente
📏 Dimensiones: 9,800 filas × 18 columnas
💾 Memoria usada: 9.95 MB


# 3. Exploración exhaustiva

In [2]:
# ===================================================================
# BLOQUE 3: ANÁLISIS EXPLORATORIO DE DATOS (EDA)
# ===================================================================
# Información general del dataset
print("="*50)
print("📊 REPORTE DE CALIDAD DE DATOS")
print("="*50)

# 1. INFORMACIÓN BÁSICA
print("\n🔢 INFORMACIÓN BÁSICA:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"Memoria total: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# 2. TIPOS DE DATOS
print("\n📋 TIPOS DE DATOS:")
print(df.dtypes.value_counts())

# 3. ANÁLISIS DE VALORES FALTANTES
print("\n❓ VALORES FALTANTES:")
missing_data = df.isnull().sum()
missing_percent = 100 * missing_data / len(df)
missing_table = pd.DataFrame({
    'Columna': missing_data.index,
    'Faltantes': missing_data.values,
    'Porcentaje': missing_percent.values
}).sort_values('Porcentaje', ascending=False)

# Mostrar solo columnas con valores faltantes
missing_table_filtered = missing_table[missing_table['Faltantes'] > 0]
if len(missing_table_filtered) > 0:
    print(missing_table_filtered.to_string(index=False))
else:
    print("✅ No hay valores faltantes en el dataset")

# 4. DUPLICADOS
print("\n🔄 DUPLICADOS:")
duplicates = df.duplicated().sum()
print(f"Filas duplicadas: {duplicates:,} ({100*duplicates/len(df):.2f}%)")

# 5. ESTADÍSTICAS DESCRIPTIVAS
print("\n📊 ESTADÍSTICAS DESCRIPTIVAS:")
print("\nColumnas numéricas:")
print(df.describe())

print("\nColumnas categóricas:")
print(df.describe(include=['object']))

📊 REPORTE DE CALIDAD DE DATOS

🔢 INFORMACIÓN BÁSICA:
Filas: 9,800
Columnas: 18
Memoria total: 9.95 MB

📋 TIPOS DE DATOS:
object     15
float64     2
int64       1
Name: count, dtype: int64

❓ VALORES FALTANTES:
    Columna  Faltantes  Porcentaje
Postal Code         11    0.112245

🔄 DUPLICADOS:
Filas duplicadas: 0 (0.00%)

📊 ESTADÍSTICAS DESCRIPTIVAS:

Columnas numéricas:
            Row ID   Postal Code         Sales
count  9800.000000   9789.000000   9800.000000
mean   4900.500000  55273.322403    230.769059
std    2829.160653  32041.223413    626.651875
min       1.000000   1040.000000      0.444000
25%    2450.750000  23223.000000     17.248000
50%    4900.500000  58103.000000     54.490000
75%    7350.250000  90008.000000    210.605000
max    9800.000000  99301.000000  22638.480000

Columnas categóricas:
              Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
count             9800        9800        9800            9800        9800   
unique            4922 

# 4. Limpieza Sistemática

In [3]:
# ===================================================================
# BLOQUE 4: LIMPIEZA COMPLETA DE DATOS
# ===================================================================
# Crear una copia para mantener los datos originales
df_original = df.copy()
df_clean = df.copy()

print("📋 INICIANDO PROCESO DE LIMPIEZA...")
print(f"📊 Dataset original: {df_clean.shape}")

# 1. ELIMINAR DUPLICADOS
print("\n🔄 ELIMINANDO DUPLICADOS:")
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
duplicates_removed = initial_rows - len(df_clean)
print(f"   • Duplicados eliminados: {duplicates_removed:,}")
print(f"   • Filas restantes: {len(df_clean):,}")

# 2. LIMPIAR NOMBRES DE COLUMNAS
print("\n📝 ESTANDARIZANDO NOMBRES DE COLUMNAS:")
print("   Nombres originales:", list(df_clean.columns[:5]), "...")

df_clean.columns = (df_clean.columns
                   .str.strip()           # Quitar espacios al inicio/final
                   .str.lower()           # Convertir a minúsculas
                   .str.replace(' ', '_')     # Espacios por guiones bajos
                   .str.replace('[^a-zA-Z0-9_]', '', regex=True)) # Solo alfanuméricos
print("   Nombres estandarizados:", list(df_clean.columns[:5]), "...")

# 3. MANEJO DE VALORES FALTANTES
print("\n🔧 MANEJO DE VALORES FALTANTES:")
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
categorical_cols = df_clean.select_dtypes(include=['object']).columns

for col in numeric_cols:
    missing_count = df_clean[col].isnull().sum()
    if missing_count > 0:
        median_value = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_value)
        print(f"   • {col}: {missing_count:,} valores → mediana ({median_value:.2f})")

for col in categorical_cols:
    missing_count = df_clean[col].isnull().sum()
    if missing_count > 0:
        mode_values = df_clean[col].mode()
        fill_value = mode_values[0] if len(mode_values) > 0 else 'Unknown'
        df_clean[col] = df_clean[col].fillna(fill_value)
        print(f"   • {col}: {missing_count:,} valores → '{fill_value}'")

# 4. DETECTAR Y TRATAR OUTLIERS
print("\n🎯 TRATAMIENTO DE OUTLIERS:")
for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers_mask = (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)
    outliers_count = outliers_mask.sum()

    if outliers_count > 0:
        outlier_percentage = (outliers_count / len(df_clean)) * 100
        if outlier_percentage > 5:  # Winsorización si >5% outliers
            df_clean[col] = np.where(df_clean[col] < lower_bound, lower_bound, df_clean[col])
            df_clean[col] = np.where(df_clean[col] > upper_bound, upper_bound, df_clean[col])
            print(f"   • {col}: {outliers_count:,} outliers winsorizados ({outlier_percentage:.1f}%)")
        else:
            print(f"   • {col}: {outliers_count:,} outliers mantenidos ({outlier_percentage:.1f}%)")

print(f"\n✅ LIMPIEZA COMPLETADA")
print(f"📊 Dimensiones finales: {df_clean.shape}")
print(f"❓ Valores faltantes restantes: {df_clean.isnull().sum().sum()}")

📋 INICIANDO PROCESO DE LIMPIEZA...
📊 Dataset original: (9800, 18)

🔄 ELIMINANDO DUPLICADOS:
   • Duplicados eliminados: 0
   • Filas restantes: 9,800

📝 ESTANDARIZANDO NOMBRES DE COLUMNAS:
   Nombres originales: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode'] ...
   Nombres estandarizados: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode'] ...

🔧 MANEJO DE VALORES FALTANTES:
   • postal_code: 11 valores → mediana (58103.00)

🎯 TRATAMIENTO DE OUTLIERS:
   • sales: 1,145 outliers winsorizados (11.7%)

✅ LIMPIEZA COMPLETADA
📊 Dimensiones finales: (9800, 18)
❓ Valores faltantes restantes: 0


# 5. Exportar dataset limpio

In [ ]:
# ===================================================================
# BLOQUE 5: EXPORTAR DATASET LIMPIO (¡IMPORTANTE PARA ACTIVIDAD 6!)
# ===================================================================
print("💾 GUARDANDO DATASET LIMPIO...")

# Opción 1: Guardar como CSV (recomendado para Power BI)
filename_clean = 'cleaned_customersales.csv'
df_clean.to_csv(filename_clean, index=False, encoding='utf-8')
print(f"✅ Dataset limpio guardado como: {filename_clean}")

# Opción 2: Descargar automáticamente a tu computadora
from google.colab import files
files.download(filename_clean)
print(f"📥 Archivo descargado a tu computadora")

# Verificación final
print(f"📊 Archivo final: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas")
print(f"📁 Tamaño: {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"🎯 Listo para usar en Actividad 6!")

💾 GUARDANDO DATASET LIMPIO...
✅ Dataset limpio guardado como: cleaned_customersales.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Archivo descargado a tu computadora
📊 Archivo final: 9,800 filas × 18 columnas
📁 Tamaño: 9.96 MB
🎯 Listo para usar en Actividad 6!


In [38]:
# ===== VALIDACIÓN INICIAL SUPERSTORE SALES =====

print("🛒 SUPERSTORE SALES - VALIDACIÓN INICIAL")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Período: {df['order_date'].min()} a {df['order_date'].max()}")

# Información básica
print(f"\n📊 OVERVIEW:")
print(f"total_orders: {df['order_id'].nunique():,}")
print(f"total_customers: {df['customer_id'].nunique():,}")
print(f"total_products: {df['product_id'].nunique():,}")
print(f"sales_range: ${df['sales'].min():.2f} - ${df['sales'].max():.2f}")

# Verificar calidad de datos
print(f"\n🔍 CALIDAD DE DATOS:")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")

# Identificar variables sospechosas
print(f"\n🔍 VARIABLES 'SOSPECHOSAS' PARA VDRER:")
print(f"row_id: Secuencial 1-{df['row_id'].max()}")
print(f"order_id: {df['order_id'].nunique()} órdenes únicas")
print(f"customer_id: Patrón {df['customer_id'].iloc[0]}")
print(f"product_id: Patrón {df['product_id'].iloc[0]}")

print(f"\n✅ Dataset listo para aplicar Framework VDRER")

🛒 SUPERSTORE SALES - VALIDACIÓN INICIAL
Shape: (9800, 18)
Período: 01/01/2018 a 31/12/2017

📊 OVERVIEW:
total_orders: 4,922
total_customers: 793
total_products: 1,861
sales_range: $0.44 - $500.64

🔍 CALIDAD DE DATOS:
Missing values: 0
Duplicates: 0

🔍 VARIABLES 'SOSPECHOSAS' PARA VDRER:
row_id: Secuencial 1-9800
order_id: 4922 órdenes únicas
customer_id: Patrón CG-12520
product_id: Patrón FUR-BO-10001798

✅ Dataset listo para aplicar Framework VDRER


# 🔍 FASE 1: VALIDAR - Verificación de Dataset Limpio -Fases del Framework VDRER

In [39]:
# ===================================================================
# 🔍 FASE 1: VALIDAR - Customer Sales Dataset desde df_clean
# ===================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*70)
print("🛒 ACTIVIDAD 6: CUSTOMER SALES - EXTRACCIÓN DE VALOR VDRER")
print("="*70)

# Asignar copia del dataset limpio
df = df_clean.copy()
print(f"✅ Dataset asignado desde df_clean: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# Convertir fechas si existen
if 'OrderDate' in df.columns:
    df['OrderDate'] = pd.to_datetime(df['OrderDate'])
if 'ShipDate' in df.columns:
    df['ShipDate'] = pd.to_datetime(df['ShipDate'])

# Overview del negocio
print(f"\n🏪 OVERVIEW CUSTOMER SALES:")
if 'OrderDate' in df.columns:
    print(f"📅 Período: {df['OrderDate'].min().strftime('%Y-%m-%d')} a {df['OrderDate'].max().strftime('%Y-%m-%d')}")
if 'ORDERNUMBER' in df.columns:
    print(f"🛍️ Órdenes únicas: {df['ORDERNUMBER'].nunique():,}")
if 'CUSTOMERNAME' in df.columns:
    print(f"👥 Clientes únicos: {df['CUSTOMERNAME'].nunique():,}")
if 'PRODUCTCODE' in df.columns:
    print(f"📦 Productos únicos: {df['PRODUCTCODE'].nunique():,}")
if 'SALES' in df.columns:
    print(f"💰 Ventas totales: ${df['SALES'].sum():,.2f}")
    print(f"📊 Venta promedio: ${df['SALES'].mean():.2f}")

# Verificación de calidad
print(f"\n🔍 CALIDAD DE DATOS:")
print("Missing values por columna:")
for col in df.columns:
    missing = df[col].isnull().sum()
    if missing > 0:
        print(f"   {col}: {missing} ({missing/len(df)*100:.1f}%)")

print(f"Duplicados: {df.duplicated().sum()}")

print("\n✅ FASE 1 COMPLETADA - Dataset Customer Sales validado desde df_clean")
print("📋 Ejecuta Fase 2 para descubrir variables 'inútiles'")



🛒 ACTIVIDAD 6: CUSTOMER SALES - EXTRACCIÓN DE VALOR VDRER
✅ Dataset asignado desde df_clean: 9,800 filas × 18 columnas

🏪 OVERVIEW CUSTOMER SALES:

🔍 CALIDAD DE DATOS:
Missing values por columna:
Duplicados: 0

✅ FASE 1 COMPLETADA - Dataset Customer Sales validado desde df_clean
📋 Ejecuta Fase 2 para descubrir variables 'inútiles'


# 🔍 FASE 2: DESCUBRIR - Identificar Variables Aparentemente "Inútiles" -Fases del Framework VDRER

In [40]:
# ===================================================================
# 🔍 FASE 2: DESCUBRIR - Variables "Inútiles" en SuperStore
# ===================================================================

print("\n🔍 FASE 2: DESCUBRIR - Analizando variables aparentemente 'inútiles'")

# ANÁLISIS 1: Row ID - ¿Solo secuencial?
print("\n📊 ANÁLISIS ROW ID:")
print(f"Rango: {df['row_id'].min()} - {df['row_id'].max()}")
print(f"¿Es secuencial? {(df['row_id'] == range(1, len(df)+1)).all()}")

# Verificar si Row ID correlaciona con fechas (orden cronológico)
df_sorted_by_date = df.sort_values('order_date')
# Convert the index to a Series before calculating correlation
correlation_with_time = df_sorted_by_date['row_id'].corr(pd.Series(df_sorted_by_date.index))
print(f"Correlación Row ID con tiempo: {correlation_with_time:.3f}")

if correlation_with_time > 0.8:
    print("💡 INSIGHT: Row ID refleja orden cronológico de entrada al sistema")
else:
    print("💡 INSIGHT: Row ID es puramente secuencial, sin información temporal")

# ANÁLISIS 2: Order ID patterns
print("\n📊 ANÁLISIS ORDER ID:")
print(f"Formato típico: {df['order_id'].iloc[0]}")
print(f"Longitud promedio: {df['order_id'].str.len().mean():.1f} caracteres")

# Extraer patrones de Order ID
order_prefixes = df['order_id'].str[:2].value_counts()
print(f"Prefijos más comunes: {order_prefixes.head().to_dict()}")

# ANÁLISIS 3: Customer ID structure
print("\n📊 ANÁLISIS CUSTOMER ID:")
print(f"Formato típico: {df['customer_id'].iloc[0]}")
print(f"Patrón: {df['customer_id'].str.len().nunique()} longitudes diferentes")

# Extraer prefijos de Customer ID
customer_prefixes = df['customer_id'].str[:2].value_counts()
print(f"Prefijos más comunes: {customer_prefixes.head().to_dict()}")

# ANÁLISIS 4: Product ID encoding
print("\n📊 ANÁLISIS PRODUCT ID:")
print(f"Formato típico: {df['product_id'].iloc[0]}")
product_prefixes = df['product_id'].str[:3].value_counts()
print(f"Prefijos de productos: {product_prefixes.head().to_dict()}")

# Verificar si prefijos correlacionan con categorías
sample_products = df[['product_id', 'category', 'subcategory']].drop_duplicates().head(10)
print(f"\nMuestra Product ID vs Category:")
print(sample_products)

print("\n✅ FASE 2 COMPLETADA - Variables 'inútiles' analizadas")
print("📋 Ejecuta Fase 3 para extraer valor de estos patrones")


🔍 FASE 2: DESCUBRIR - Analizando variables aparentemente 'inútiles'

📊 ANÁLISIS ROW ID:
Rango: 1 - 9800
¿Es secuencial? True
Correlación Row ID con tiempo: 0.006
💡 INSIGHT: Row ID es puramente secuencial, sin información temporal

📊 ANÁLISIS ORDER ID:
Formato típico: CA-2017-152156
Longitud promedio: 14.0 caracteres
Prefijos más comunes: {'CA': 8161, 'US': 1639}

📊 ANÁLISIS CUSTOMER ID:
Formato típico: CG-12520
Patrón: 1 longitudes diferentes
Prefijos más comunes: {'SC': 204, 'CS': 139, 'DB': 135, 'TB': 131, 'CC': 129}

📊 ANÁLISIS PRODUCT ID:
Formato típico: FUR-BO-10001798
Prefijos de productos: {'OFF': 5909, 'FUR': 2078, 'TEC': 1813}

Muestra Product ID vs Category:
        product_id         category  subcategory
0  FUR-BO-10001798        Furniture    Bookcases
1  FUR-CH-10000454        Furniture       Chairs
2  OFF-LA-10000240  Office Supplies       Labels
3  FUR-TA-10000577        Furniture       Tables
4  OFF-ST-10000760  Office Supplies      Storage
5  FUR-FU-10001487        Fu

In [41]:
print(df.columns.tolist())


['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'subcategory', 'product_name', 'sales']


# ♻️ FASE 3: REUTILIZAR - Extraer Valor de Variables "Inútiles" -Fases del Framework VDRER

In [14]:
# ===================================================================
# ♻️ FASE 3: REUTILIZAR - Extraer Valor de Variables "Inútiles"
# ===================================================================

print("\n⚙️ FASE 3: REUTILIZAR - Extrayendo valor de IDs y códigos")

insights_found = []

# 💎 INSIGHT 1: Row ID como proxy temporal (si existe correlación)
print("\n💎 INSIGHT 1: Row ID y Timing de Negocios")
try:
    # Crear ventanas temporales basadas en Row ID
    df['row_id_quintile'] = pd.qcut(df['row_id'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

    # Analizar ventas por quintil de entrada
    sales_by_quintile = df.groupby('row_id_quintile')['sales'].agg(['mean', 'count', 'sum'])

    print("📈 Ventas por quintil de Row ID (cronología de entrada):")
    for quintile in sales_by_quintile.index:
        avg_sales = sales_by_quintile.loc[quintile, 'mean']
        total_sales = sales_by_quintile.loc[quintile, 'sum']
        print(f"   {quintile}: Promedio ${avg_sales:.2f}, Total ${total_sales:,.2f}")

    insights_found.append("Row ID revela patterns de performance por período de entrada")
except Exception as e:
    print(f"⚠️ Error en análisis Row ID: {e}")

# 💎 INSIGHT 2: Order ID prefix patterns
print("\n💎 INSIGHT 2: Order ID - Patrones de Sistema/Región")
try:
    df['order_prefix'] = df['order_id'].str[:2]

    # Analizar ventas por prefijo
    prefix_analysis = df.groupby('order_prefix').agg({
        'sales': ['mean', 'sum', 'count'],
        'region': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'
    }).round(2)

    print("🎯 Top 5 prefijos de Order ID por ventas:")
    top_prefixes = df.groupby('order_prefix')['sales'].sum().sort_values(ascending=False).head()
    for prefix, sales in top_prefixes.items():
        region = df[df['order_prefix'] == prefix]['region'].mode().iloc[0]
        print(f"   Prefix '{prefix}': ${sales:,.2f} total, Región dominante: {region}")

    insights_found.append("Order ID prefixes correlacionan con regiones geográficas")
except Exception as e:
    print(f"⚠️ Error en análisis Order ID: {e}")

# 💎 INSIGHT 3: Customer ID encoding de segmentos
print("\n💎 INSIGHT 3: Customer ID - Segmentación Oculta")
try:
    df['customer_prefix'] = df['customer_id'].str[:2]

    # Analizar segmentos por prefijo de Customer ID
    customer_analysis = df.groupby(['customer_prefix', 'segment']).size().unstack(fill_value=0)

    print("👥 Distribución de segmentos por prefijo de Customer ID:")
    for prefix in customer_analysis.index[:5]:  # Top 5
        total = customer_analysis.loc[prefix].sum()
        if total > 0:
            dominant_segment = customer_analysis.loc[prefix].idxmax()
            percentage = customer_analysis.loc[prefix].max() / total * 100
            print(f"   Prefix '{prefix}': {total} customers, {percentage:.1f}% {dominant_segment}")

    insights_found.append("Customer ID prefixes reflejan estrategias de segmentación")
except Exception as e:
    print(f"⚠️ Error en análisis Customer ID: {e}")

# 💎 INSIGHT 4: Product ID hierarchy
print("\n💎 INSIGHT 4: Product ID - Jerarquía de Categorías")
try:
    df['product_prefix'] = df['product_id'].str[:3]

    # Verificar si prefijos correlacionan con categorías
    product_category_map = df.groupby('product_prefix')['category'].apply(
        lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Mixed'
    )

    print("📦 Product ID prefixes por categoría:")
    for category in df['category'].unique():
        matching_prefixes = product_category_map[product_category_map == category]
        if len(matching_prefixes) > 0:
            print(f"   {category}: prefixes {list(matching_prefixes.index[:3])}")  # Top 3

    insights_found.append("Product ID structure codifica jerarquías de categorías")
except Exception as e:
    print(f"⚠️ Error en análisis Product ID: {e}")

print(f"\n✅ FASE 3 COMPLETADA - {len(insights_found)} insights valiosos extraídos")
print("📋 Ejecuta Fase 4 para crear features derivadas")


⚙️ FASE 3: REUTILIZAR - Extrayendo valor de IDs y códigos

💎 INSIGHT 1: Row ID y Timing de Negocios
📈 Ventas por quintil de Row ID (cronología de entrada):
   Q1: Promedio $141.08, Total $276,520.55
   Q2: Promedio $145.73, Total $285,636.15
   Q3: Promedio $133.31, Total $261,293.83
   Q4: Promedio $145.47, Total $285,127.74
   Q5: Promedio $138.48, Total $271,416.47

💎 INSIGHT 2: Order ID - Patrones de Sistema/Región
🎯 Top 5 prefijos de Order ID por ventas:
   Prefix 'CA': $1,150,892.56 total, Región dominante: West
   Prefix 'US': $229,102.19 total, Región dominante: West

💎 INSIGHT 3: Customer ID - Segmentación Oculta
👥 Distribución de segmentos por prefijo de Customer ID:
   Prefix 'AA': 56 customers, 100.0% Consumer
   Prefix 'AB': 92 customers, 56.5% Consumer
   Prefix 'AC': 38 customers, 60.5% Corporate
   Prefix 'AD': 12 customers, 100.0% Home Office
   Prefix 'AF': 23 customers, 100.0% Consumer

💎 INSIGHT 4: Product ID - Jerarquía de Categorías
📦 Product ID prefixes por cate

# ⚙️ FASE 4: ENGINEER - Crear Features Derivadas con Valor -Fases del Framework VDRER

In [43]:
# ===================================================================
# ⚙️ FASE 4: ENGINEER - Crear Features Derivadas con Valor
# ===================================================================

# Corregir formatos antes de Fase 4 (con dayfirst para DD/MM/YYYY)
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True)
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=True)
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')

# Asegurar que columnas clave tengan el tipo correcto
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True)
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=True)
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')


print("\n🔧 FASE 4: ENGINEER - Creando features valiosas para retail")

new_features_created = []

# Feature 1: Customer Lifetime Value (CLV simple anualizado)
try:
    clv = df.groupby('customer_id').agg({
        'sales': 'sum',
        'order_date': lambda x: (x.max() - x.min()).days / 365 + 0.01  # evitar división por cero
    }).rename(columns={'sales': 'total_sales', 'order_date': 'years_active'})

    clv['clv_simple'] = clv['total_sales'] / clv['years_active']
    clv.reset_index(inplace=True)

    # Evitar colisión de nombre si ya existe
    if 'clv_simple' in df.columns:
        df.drop(columns=['clv_simple'], inplace=True)

    df = df.merge(clv[['customer_id', 'clv_simple']], on='customer_id', how='left')

    print("✅ customer_lifetime_value: CLV anualizado por cliente")
    print(f"   Media CLV: ${df['clv_simple'].mean():.2f}")
    print(f"   Top CLV: ${df['clv_simple'].max():,.2f}")
    new_features_created.append('clv_simple')
except Exception as e:
    print(f"⚠️ Error creando CLV: {e}")

# Feature 2: Product Performance Index
# Eliminar si ya existe la columna antes de recrearla
if 'product_performance_index' in df.columns:
    df = df.drop(columns=['product_performance_index'])

# Crear de nuevo la feature
product_perf = df.groupby('product_id').agg({
    'sales': 'sum',
    'order_id': 'nunique'
}).rename(columns={'sales': 'total_sales', 'order_id': 'order_count'})

product_perf['product_performance_index'] = (
    product_perf['total_sales'] / product_perf['order_count']
).round(2)

try:
    df = df.merge(product_perf[['product_performance_index']],
                  left_on='product_id', right_index=True, how='left')
    print("✅ product_performance_index: Índice combinado ventas x frecuencia")
    new_features_created.append('product_performance_index')
except Exception as e:
    print(f"⚠️ Error creando Product Performance Index: {e}")

# Feature 3: Geographic Revenue Density
try:
    geo_metrics = df.groupby(['state', 'region']).agg({
        'sales': 'sum',
        'customer_id': 'nunique'
    }).reset_index()

    geo_metrics['revenue_per_customer'] = geo_metrics['sales'] / geo_metrics['customer_id']

    if 'revenue_per_customer' in df.columns:
        df.drop(columns=['revenue_per_customer'], inplace=True)

    df = df.merge(geo_metrics[['state', 'region', 'revenue_per_customer']],
                  on=['state', 'region'], how='left')

    print("✅ geographic_revenue_density: Revenue por customer por estado")
    print(f"   Media: ${df['revenue_per_customer'].mean():.2f}")
    new_features_created.append('revenue_per_customer')
except Exception as e:
    print(f"⚠️ Error creando Geographic Revenue Density: {e}")

# Feature 4: Order Timing Intelligence
try:
    df['order_to_ship_days'] = (df['ship_date'] - df['order_date']).dt.days
    df['order_day_of_week'] = df['order_date'].dt.day_name()
    df['order_month'] = df['order_date'].dt.month
    df['order_quarter'] = df['order_date'].dt.quarter
    df['shipping_efficiency'] = np.where(df['order_to_ship_days'] <= 2, 'Fast',
                                np.where(df['order_to_ship_days'] <= 5, 'Normal', 'Slow'))

    print("✅ order_timing_features: Análisis temporal de órdenes")
    print(f"   Promedio días order-to-ship: {df['order_to_ship_days'].mean():.1f}")
    shipping_dist = df['shipping_efficiency'].value_counts()
    print(f"   Distribución shipping: {shipping_dist.to_dict()}")
    new_features_created.extend(['order_to_ship_days', 'shipping_efficiency'])
except Exception as e:
    print(f"⚠️ Error creando Order Timing features: {e}")

# Feature 5: Customer Segmentation (VDRER)
try:
    df['customer_value_score'] = pd.qcut(df['clv_simple'], q=3, labels=['Bronze', 'Silver', 'Gold'])
    df['customer_segment_vdrer'] = df['customer_value_score']

    print("✅ customer_segmentation: Segmentación automática basada en valor")
    print(f"   Distribución: {df['customer_segment_vdrer'].value_counts().to_dict()}")
    new_features_created.extend(['customer_value_score', 'customer_segment_vdrer'])
except Exception as e:
    print(f"⚠️ Error creando Customer Segmentation: {e}")

print(f"\n✅ FASE 4 COMPLETADA - {len(new_features_created)} features creadas")
print(f"📊 Features: {new_features_created}")
print("📋 Ejecuta Fase 5 para reportar y exportar")



🔧 FASE 4: ENGINEER - Creando features valiosas para retail
✅ customer_lifetime_value: CLV anualizado por cliente
   Media CLV: $996.05
   Top CLV: $104,304.10
✅ product_performance_index: Índice combinado ventas x frecuencia
✅ geographic_revenue_density: Revenue por customer por estado
   Media: $357.61
✅ order_timing_features: Análisis temporal de órdenes
   Promedio días order-to-ship: 4.0
   Distribución shipping: {'Normal': 5843, 'Fast': 2172, 'Slow': 1785}
✅ customer_segmentation: Segmentación automática basada en valor
   Distribución: {'Bronze': 3270, 'Silver': 3267, 'Gold': 3263}

✅ FASE 4 COMPLETADA - 7 features creadas
📊 Features: ['clv_simple', 'product_performance_index', 'revenue_per_customer', 'order_to_ship_days', 'shipping_efficiency', 'customer_value_score', 'customer_segment_vdrer']
📋 Ejecuta Fase 5 para reportar y exportar


# 📋 FASE 5: REPORTAR - Documentar Insights y Exportar Dataset -Fases del Framework VDRER

In [44]:
# ===================================================================
# 📋 FASE 5: REPORTAR - Documentar y Exportar Superstore Enriquecido
# ===================================================================

print("\n📋 FASE 5: REPORTAR - Consolidando insights Superstore")

# Resumen ejecutivo del análisis
print(f"\n📊 REPORTE FINAL - FRAMEWORK VDRER SUPERSTORE:")
print(f"   🛒 Dataset original: {df.shape[0]:,} órdenes × {18} columnas")
print(f"   💎 Insights de 'deshechos': {len(insights_found) if 'insights_found' in locals() else 'N/A'}")
print(f"   ⚙️ Features creadas: {len(new_features_created) if 'new_features_created' in locals() else 'N/A'}")
print(f"   📈 Dimensiones finales: {df.shape}")

# Análisis de valor generado por features
print(f"\n📈 VALOR GENERADO POR NUEVAS FEATURES:")
if 'new_features_created' in locals():
    for feature in new_features_created[:5]:  # Top 5
        if feature in df.columns:
            if df[feature].dtype in ['float64', 'int64']:
                print(f"   • {feature}: min={df[feature].min():.2f}, max={df[feature].max():.2f}")
            else:
                print(f"   • {feature}: {df[feature].nunique()} categorías únicas")

# Top insights por categoría
print(f"\n🏆 TOP INSIGHTS POR CATEGORÍA DE ANÁLISIS:")
print(f"   💳 Customer Intelligence:")
print(f"     - CLV promedio: ${df['clv_simple'].mean():.2f}" if 'clv_simple' in df.columns else "     - CLV: No calculado")
print(f"     - Segmentación: {df['customer_segment_vdrer'].value_counts().to_dict()}" if 'customer_segment_vdrer' in df.columns else "     - Segmentación: No aplicada")

print(f"   📦 Product Intelligence:")
performance_top = df.nlargest(3, 'product_performance_index')[['product_name', 'product_performance_index']] if 'product_performance_index' in df.columns else None
if performance_top is not None:
    print(f"     - Top productos por performance:")
    for _, row in performance_top.iterrows():
        print(f"       • {row['product_name'][:30]}... ({row['product_performance_index']:.3f})")

print(f"   🌍 Geographic Intelligence:")
if 'revenue_per_customer' in df.columns:
    geo_top = df.groupby('state')['revenue_per_customer'].first().nlargest(3)
    print(f"     - Estados top por revenue/customer:")
    for state, revenue in geo_top.items():
        print(f"       • {state}: ${revenue:.2f}")

# ROI estimado del análisis
print(f"\n💰 ESTIMACIÓN DE ROI - FRAMEWORK VDRER:")
print(f"   📊 Variables 'inútiles' analizadas: 4 (row_id, order_id, customer_id, product_id)")
print(f"   💡 Insights de negocio extraídos: {len(insights_found) if 'insights_found' in locals() else 0}")
print(f"   🎯 Features accionables: {len(new_features_created) if 'new_features_created' in locals() else 0}")
print(f"   📈 Preparación para KPIs: customer, product, geographic, temporal")

# Exportar dataset enriquecido
print(f"\n💾 EXPORTANDO DATASET SUPERSTORE ENRIQUECIDO...")
try:
    filename_enriched = 'superstore_value_extracted_python.csv'

    # Copiar dataframe y convertir columnas datetime a string
    export_df = df.copy()

    # Convertir columnas datetime64[ns] a string (formato ISO)
    datetime_cols = export_df.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns
    for col in datetime_cols:
        export_df[col] = export_df[col].dt.strftime('%Y-%m-%d')

    # Guardar CSV
    export_df.to_csv(filename_enriched, index=False, encoding='utf-8')
    print(f"✅ Guardado como: {filename_enriched}")

    # Intentar descarga automática (solo si estás en Google Colab)
    try:
        from google.colab import files
        files.download(filename_enriched)
        print(f"📥 ¡Archivo descargado automáticamente!")
    except:
        print(f"📝 Archivo listo para descargar manualmente")

except Exception as e:
    print(f"⚠️ Error en exportación: {e}")

# Summary para Power BI
print(f"\n📊 RESUMEN PARA DASHBOARD POWER BI:")
print(f"   🏠 Página 1 - Overview: ventas totales, customer count, product performance")
print(f"   📈 Página 2 - Customer Intelligence: CLV, segmentación, geographic patterns")
print(f"   📦 Página 3 - Product Intelligence: performance index, category analysis")
print(f"   💎 Página 4 - Valor VDRER: insights de variables 'inútiles' documentados")

print("\n" + "="*70)
print("🎉 SUPERSTORE VDRER COMPLETADO - DATASET LISTO PARA POWER BI")
print("🚀 ¡Datos de retail transformados en inteligencia accionable!")
print("="*70)

# Mensaje de aprendizaje final
print(f"\n📚 REFLEXIÓN RETAIL INTELLIGENCE:")
print(f"✅ Has aplicado VDRER exitosamente a datos de e-commerce")
print(f"✅ Transformaste IDs 'inútiles' en customer & product intelligence")
print(f"✅ Creaste features de CLV, performance y segmentación")
print(f"✅ Preparaste base sólida para retail analytics dashboard")
print(f"\n💡 ¡Has convertido códigos aparentemente irrelevantes en oro de retail!")



📋 FASE 5: REPORTAR - Consolidando insights Superstore

📊 REPORTE FINAL - FRAMEWORK VDRER SUPERSTORE:
   🛒 Dataset original: 9,800 órdenes × 18 columnas
   💎 Insights de 'deshechos': 4
   ⚙️ Features creadas: 7
   📈 Dimensiones finales: (9800, 28)

📈 VALOR GENERADO POR NUEVAS FEATURES:
   • clv_simple: min=3.26, max=104304.10
   • product_performance_index: min=1.62, max=510.13
   • revenue_per_customer: min=178.25, max=582.10
   • order_to_ship_days: min=0.00, max=7.00
   • shipping_efficiency: 3 categorías únicas

🏆 TOP INSIGHTS POR CATEGORÍA DE ANÁLISIS:
   💳 Customer Intelligence:
     - CLV promedio: $996.05
     - Segmentación: {'Bronze': 3270, 'Silver': 3267, 'Gold': 3263}
   📦 Product Intelligence:
     - Top productos por performance:
       • Adjustable Depth Letter/Legal ... (510.130)
       • Adjustable Depth Letter/Legal ... (510.130)
       • Adjustable Depth Letter/Legal ... (510.130)
   🌍 Geographic Intelligence:
     - Estados top por revenue/customer:
       • Vermont

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 ¡Archivo descargado automáticamente!

📊 RESUMEN PARA DASHBOARD POWER BI:
   🏠 Página 1 - Overview: ventas totales, customer count, product performance
   📈 Página 2 - Customer Intelligence: CLV, segmentación, geographic patterns
   📦 Página 3 - Product Intelligence: performance index, category analysis
   💎 Página 4 - Valor VDRER: insights de variables 'inútiles' documentados

🎉 SUPERSTORE VDRER COMPLETADO - DATASET LISTO PARA POWER BI
🚀 ¡Datos de retail transformados en inteligencia accionable!

📚 REFLEXIÓN RETAIL INTELLIGENCE:
✅ Has aplicado VDRER exitosamente a datos de e-commerce
✅ Transformaste IDs 'inútiles' en customer & product intelligence
✅ Creaste features de CLV, performance y segmentación
✅ Preparaste base sólida para retail analytics dashboard

💡 ¡Has convertido códigos aparentemente irrelevantes en oro de retail!
